## 第 4 课：Tiled MMA 与 GEMM 三级 Tiling

> 对应原文：笔记 (3)。把单指令 MMA 扩展为单个 Tile（32×32×16，256 线程）的运算，并建立 GEMM 三级 Tiling 的概念模型。

## 学习目标

- 建立算子优化方法论：**计算 / 通信 / 存储**三维度；
- 理解 GEMM **三级 Tiling**（Global → Block → Tile → MMA Atom）及各自对应的硬件特性；
- 掌握 `make_tiled_mma` 的两个新参数：**MMAThrLayout**（线程扩展）与 **MMATileLayout**（指令扩展）；
- 会读 partition 后 Tensor 的 metadata（print 输出）。

## 1. 算子优化方法论

| 维度 | 关注指标 | 关键点 |
|---|---|---|
| 计算 | FLOPS、Tensor Core 利用率 | 堆满 mma 指令、从算法层减少 FLOPs |
| 通信 | latency | 减少高延迟通信量，或用流水线掩盖 |
| 存储 | Occupancy（SMEM/寄存器利用率） | 在增加 Occupancy 的同时避免副作用 |

注意：**FLOPS 低不代表计算需要优化**，很多时候是通信/存储瓶颈导致 Tensor Core 无事可做。当前大部分算子在 Tensor Core 面前都是 memory bound。

存储资源上限（SM90/SM100）：一个 thread block 最多申请 **227 KB SMEM**、最多 **64K 个寄存器**，每线程最多 255 个寄存器。寄存器溢出 → Local Memory（访存效率约等于 GMEM）；SMEM 溢出 → 无法运行；Occupancy 不足 → 有访存效率优化空间。

![单指令扩展至任意规模的矩阵运算](assets/figs/fig_01_单指令扩展至任意规模的矩阵运算.png)

## 2. GEMM 三级 Tiling

只做一级 Tiling（单 block 循环）的问题：没有利用多 SM 并行。

- **第一级 Global → Block**：把 D 切成 16×8 的 tile，每个 tile 交给一个 block，SM 间并行；block 内沿 K 循环；
- **第二级 Block → Tile**：扩展线程数（更多 warp）+ 每 warp 多发 mma 指令（如 32×32×16，256 线程）。限制：block ≤ 2048 线程；寄存器扩展过大会 register spilling；
- **第三级 Tile → MMA Atom**：受 block SMEM 限制，沿 K 分块循环拷贝 A/B 分块，前一轮结果累加。

![GEMM 三级 Tiling](assets/figs/fig_06_GEMM_三级_Tiling.png)

每一级和硬件特性强相关：

In [ ]:
Global → Block ：多个 SM 并行计算
Block  → Tile ：利用 SMEM 低延迟，减少 GMEM 访存量（数据复用）
Tile   → MMA Atom ：填满多核 Tensor Core（每个 Tile 含足够多 mma 指令）

硬件更新会改变 Tiling 方式，例如 Blackwell 的 2SM MMA 利用 Distributed SMEM，增加 Cluster 层面的 Tiling。

![Tile Tiling](assets/figs/fig_04_Tile_Tiling.png)  ![Block Tiling](assets/figs/fig_05_Block_Tiling.png)

## 3. Tiled MMA 实现

本篇规格：单指令 16×8×8，Tile 扩展到 32×32×16，线程 32 → 256。

### 3.1 make_tiled_mma API

新增两个参数：`MMAThrLayout`（线程扩展）和 `MMATileLayout`（指令扩展 / Tile 总规模）：

In [ ]:
using MMA_op = SM80_16x8x8_F32BF16BF16F32_TN;
using MMA_traits = MMA_Traits<MMA_op>;
using MMA_shape = MMA_traits::Shape_MNK;

static constexpr int kMmaThrExpandM = 2;   // M 维线程扩展 2 倍
static constexpr int kMmaThrExpandN = 4;   // N 维线程扩展 4 倍
static constexpr int kMmaThrExpandK = 1;   // K 维不扩展线程

static constexpr int kMmaValExpandM = 1;
static constexpr int kMmaValExpandN = 1;
static constexpr int kMmaValExpandK = 2;   // K 维指令扩展 2 倍

static constexpr int kMmaTileM = kMmaThrExpandM * kMmaValExpandM * get<0>(MMA_shape{});
static constexpr int kMmaTileN = kMmaThrExpandN * kMmaValExpandN * get<1>(MMA_shape{});
static constexpr int kMmaTileK = kMmaThrExpandK * kMmaValExpandK * get<2>(MMA_shape{});

using MMAThrLayout = decltype(make_layout(make_shape(Int<kMmaThrExpandM>{},
                                                     Int<kMmaThrExpandN>{},
                                                     Int<kMmaThrExpandK>{})));
using MMATileLayout = Tile<Int<kMmaTileM>, Int<kMmaTileN>, Int<kMmaTileK>>;
using TiledMMA = decltype(make_tiled_mma(MMA_op{}, MMAThrLayout{}, MMATileLayout{}));

![make_tiled_mma API 图解](assets/figs/fig_08_make_tiled_mma_API_图解.png)

**MMAThrLayout**：`(M,N,K) 坐标 -> warp_idx` 的映射，决定哪个 warp 算哪个 MMA Atom。本例 `(2,4,1):(1,2,8)`，`warp_idx = m×1 + n×2 + k×8`。例如 `(M,N,K)=(1,2,0)` → warp 5（T160~T191）。

**MMATileLayout**：M/N/K 三个维度的排列 Layout，调整它可以改变 MMA Atom 的排列次序——这就是四大 Layout 的最后一种 **Permutation Layout**（old_index -> new_index 的映射）。

扩展后 copy / gemm API 不用变，它们会根据 TiledMMA 自动处理 Atom 循环。

![Tiled MMA 图示](assets/figs/fig_09_Tiled_MMA_图示.png)

### 3.2 Tensor Metadata 详解

partition 后的 `(MMA, MMA_M, MMA_K)` 维度中，`MMA_M/N/K` 就是 mma 指令的扩展维度（= kMmaValExpand*）。打印六个 Tensor：

In [ ]:
gmem_ptr[16b](0x...) o ((_2,_2),_1,_2):((_1,128),_0,_8)   // tCgA: MMA=4, MMA_M=1, MMA_K=2
gmem_ptr[16b](0x...) o (_2,_1,_2):(_1,_0,_8)              // tCgB: MMA=2, MMA_N=1, MMA_K=2
gmem_ptr[32b](0x...) o ((_2,_2),_1,_1):((_1,256),_0,_0)   // tCgC: MMA=4, MMA_M=1, MMA_N=1
ptr[16b](0x...) o ((_2,_2),_1,_2):((_1,_2),_0,_4)         // tCrA: 寄存器版，步长紧凑
ptr[16b](0x...) o (_2,_1,_2):(_1,_0,_2)                   // tCrB
ptr[32b](0x...) o ((_2,_2),_1,_1):((_1,_2),_0,_0)         // tCrC

解读格式：`<存储介质>[<位宽>](<基地址>) o <Shape>:<Stride>`。全局内存 stride 反映矩阵跨度（128、256），寄存器 stride 是紧凑小整数（2、4）——寄存器中数据被重新紧凑排列。

这里 `MMA_M/N/K = (1, 1, 2)` 恰好等于 `kMmaValExpandM/N/K`；而 `kMmaThrExpand*` 通过增加 warp 覆盖更多输出元素，不体现在 `MMA_*` 维度上。C 矩阵没有 MMA_K 维度：K 是规约维度，两条 K 方向的 mma 都写到同一组 accumulator。

### 3.3 SASS 分析（kMmaValExpandK=2）

LDG 从 3 条变 6 条（两套 A/B 数据）；HMMA 变 2 条，且第二条的 C 用第一条的输出完成 K 归约：

In [ ]:
LDG.E R4, ...   // A[K=0] reg1
LDG.E R5, ...   // B[K=0] reg1
LDG.E R6, ...   // A[K=1] reg1
LDG.E R11, ...  // A[K=1] reg2  (+0x10 跳过第一个 atom)
LDG.E R2, ...   // A[K=0] reg2
LDG.E R3, ...   // B[K=1] reg2

HMMA.1688.F32.BF16 R4, R4, R6, RZ    // D = A[K=0]×B[K=0] + 0
HMMA.1688.F32.BF16 R4, R2, R11, R4   // D = A[K=1]×B[K=1] + D（完成 K 规约）

F2FP.BF16.F32.PACK_AB R5, R5, R4
F2FP.BF16.F32.PACK_AB R7, R7, R6
STG.E desc[UR4][R12.64], R5
STG.E desc[UR4][R2.64],  R7

注意：访存指令前面都带 `!`，提示多读了 GMEM 数据——下一篇笔记深入分析这个访存问题。

![Tiled MMA 的部分 SASS code](assets/figs/fig_10_Tiled_MMA_的部分_SASS_code.png)

## 同时回答

1. GEMM 三级 Tiling 是哪三级？每一级分别对应什么硬件特性？
2. 为什么 `MMAThrLayout` 的 K 维度一般设为 1（不在 K 维扩展线程）？K 方向的扩展由谁承担、怎么做到无需线程间通信？
3. `kMmaThrExpand` 和 `kMmaValExpand` 的区别是什么？partition 后 Tensor 的 `(MMA_M, MMA_N, MMA_K)` 等于哪个扩展系数？为什么 C 矩阵没有 MMA_K 维度？

把代码和三个答案发给我，我继续审查。